In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
data=pd.read_csv("Social_Network_Ads.csv")

In [3]:
data.head()

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0


In [4]:
data.isnull().sum()

User ID            0
Gender             0
Age                0
EstimatedSalary    0
Purchased          0
dtype: int64

In [5]:
data=data.drop("User ID",axis=1)

In [6]:
data=pd.get_dummies(data,drop_first=True)

In [7]:
data=data.astype(int)

In [8]:
data.head()

,Age,EstimatedSalary,Purchased,Gender_Male
0,19,19000,0,1
1,35,20000,0,1
2,26,43000,0,0
3,27,57000,0,0
4,19,76000,0,1


In [9]:
indep=data[["Age","EstimatedSalary","Gender_Male"]]
dep=data[["Purchased"]]

In [10]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(indep,dep,test_size=1/3,random_state=0)

In [11]:
x_train,x_test,y_train,y_test

(     Age  EstimatedSalary  Gender_Male
 218   46            96000            0
 101   28            59000            1
 311   39            96000            1
 194   28            89000            1
 326   41            72000            1
 ..   ...              ...          ...
 323   48            30000            0
 192   29            43000            1
 117   36            52000            1
 47    27            54000            0
 172   26           118000            0
 
 [266 rows x 3 columns],
      Age  EstimatedSalary  Gender_Male
 132   30            87000            1
 309   38            50000            0
 341   35            75000            1
 196   30            79000            0
 246   35            50000            0
 ..   ...              ...          ...
 168   29           148000            1
 150   26            15000            0
 393   60            42000            1
 66    24            19000            1
 240   42           149000            1
 
 [134 rows 

In [12]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
x_train=sc.fit_transform(x_train)
x_test=sc.transform(x_test)

In [13]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import BernoulliNB

# Create a BernoulliNB classifier
clf = BernoulliNB()

# Define the grid of parameters to search over
param_grid = {
  'alpha': [0.1, 0.5, 1.0]
}

# Create a GridSearchCV object
grid = GridSearchCV(clf, param_grid, cv=5)

# Fit the grid search object to the data
grid.fit(x_train, y_train)


C:\Users\USER\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1184: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\USER\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1184: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\USER\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1184: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\USER\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1184: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), 

GridSearchCV(cv=5, estimator=BernoulliNB(),
             param_grid={'alpha': [0.1, 0.5, 1.0]})

In [15]:
# Get the best performing model
best_model = grid.best_estimator_


In [16]:
# Print the best parameters
print(best_model.get_params())

{'alpha': 0.1, 'binarize': 0.0, 'class_prior': None, 'fit_prior': True, 'force_alpha': 'warn'}


In [17]:
re=grid.cv_results_

In [18]:
grid_predictions=grid.predict(x_test)

In [19]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test,grid_predictions)
        

In [20]:
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)

In [21]:
print(clf_report)

              precision    recall  f1-score   support

           0       0.76      0.92      0.83        85
           1       0.77      0.49      0.60        49

    accuracy                           0.76       134
   macro avg       0.77      0.70      0.71       134
weighted avg       0.76      0.76      0.75       134



In [22]:
print(cm)

[[78  7]
 [25 24]]


In [23]:
from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)


The f1_macro value for best parameter {'alpha': 0.1}: 0.7457605589075896


In [24]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test,grid.predict_proba(x_test)[:,1])


0.8474189675870348

In [25]:
table=pd.DataFrame.from_dict(re)

In [26]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_alpha,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.015620,0.031240,0.007180,0.007491,0.1,{'alpha': 0.1},0.740741,0.735849,0.566038,0.754717,0.735849,0.706639,0.070639,1
1,0.006518,0.007995,0.000413,0.000827,0.5,{'alpha': 0.5},0.740741,0.735849,0.566038,0.754717,0.735849,0.706639,0.070639,1
2,0.003127,0.006253,0.003124,0.006247,1.0,{'alpha': 1.0},0.740741,0.735849,0.566038,0.754717,0.735849,0.706639,0.070639,1


In [43]:
age=float(input("Age:"))
Estimated_salary=float(input("Estimated_salary:"))
sex_male=int(input("Sex Male 0 or 1:"))


Age:67
Estimated_salary:5000
Sex Male 0 or 1:0


In [44]:
Future_Prediction=grid.predict([[age,Estimated_salary,sex_male]])
print("Future_Prediction={}".format(Future_Prediction))

Future_Prediction=[1]
